In [29]:
# ============================================
# Load trained Pacman/Ghost policy and watch game
# For a separate Colab notebook
# ============================================

import random
import time
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
from google.colab import drive
from IPython.display import clear_output

drive.mount('/content/drive')


# =========================================================
# 1. ENV
# =========================================================

class TagMARLEnv:
    ACTIONS = {
        0: (-1, 0),   # UP
        1: (1, 0),    # DOWN
        2: (0, -1),   # LEFT
        3: (0, 1),    # RIGHT
        4: (0, 0),    # STAY
    }

    RAYS = [
        (-1, 0), (-1, 1), (0, 1), (1, 1),
        (1, 0), (1, -1), (0, -1), (-1, -1),
    ]

    def __init__(
        self,
        max_steps=200,
        seed=42,
        min_start_dist=4,
        pacman_speed=2,
        ghost_speed=1
    ):
        self.max_steps = max_steps
        self.rng = random.Random(seed)
        self.min_start_dist = min_start_dist
        self.pacman_speed = pacman_speed
        self.ghost_speed = ghost_speed

        self.raw_map = [
            "###########",
            "#000000000#",
            "#000#00000#",
            "#000#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00000#",
            "#000000000#",
            "###########",
        ]

        self.H = len(self.raw_map)
        self.W = len(self.raw_map[0])

        self.empty_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if self.raw_map[r][c] == "0"
        ]

        self.reset()

    def reset(self):
        self.grid = [list(row) for row in self.raw_map]

        while True:
            sampled_positions = self.rng.sample(self.empty_cells, 3)
            pacman = sampled_positions[0]
            ghosts = sampled_positions[1:]

            if all(self.manhattan(pacman, g) >= self.min_start_dist for g in ghosts):
                self.pacman = pacman
                self.ghosts = ghosts
                break

        self.steps = 0
        self.done = False
        return self.get_obs()

    def in_bounds(self, r, c):
        return 0 <= r < self.H and 0 <= c < self.W

    def move(self, pos, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = pos[0] + dr, pos[1] + dc

        if not self.in_bounds(nr, nc):
            return pos
        if self.grid[nr][nc] == "#":
            return pos
        return (nr, nc)

    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def pacman_obs(self):
        px, py = self.pacman
        feat = []
        scale_hw = max(self.H, self.W)

        for dr, dc in self.RAYS:
            wall_dist = 0
            ghost_found = False
            ghost_dist = 0
            ghost_dx = 0
            ghost_dy = 0
            ghost_mask = 0

            r, c = px, py
            while True:
                r += dr
                c += dc
                wall_dist += 1

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break

                if not ghost_found:
                    for g in self.ghosts:
                        if (r, c) == g:
                            ghost_found = True
                            ghost_dist = wall_dist
                            ghost_dx = g[0] - px
                            ghost_dy = g[1] - py
                            ghost_mask = 1
                            break

            feat.extend([
                wall_dist / scale_hw,
                ghost_dist / scale_hw,
                ghost_dx / self.H,
                ghost_dy / self.W,
                float(ghost_mask),
            ])

        return np.array(feat, dtype=np.float32)

    def ghost_obs_single(self, idx):
        gx, gy = self.ghosts[idx]
        other_ghosts = [self.ghosts[j] for j in range(len(self.ghosts)) if j != idx]
        other_sees = any(self._ghost_can_see_pacman(og) for og in other_ghosts)

        feat = []
        scale_hw = max(self.H, self.W)

        for dr, dc in self.RAYS:
            wall_dist = 0
            pacman_found = False
            pacman_dist = 0
            pacman_dx = 0
            pacman_dy = 0
            source = 0

            r, c = gx, gy
            while True:
                r += dr
                c += dc
                wall_dist += 1

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break

                if (r, c) == self.pacman and not pacman_found:
                    pacman_found = True
                    pacman_dist = wall_dist
                    pacman_dx = self.pacman[0] - gx
                    pacman_dy = self.pacman[1] - gy
                    source = 1
                    break

            if source == 0 and other_sees:
                pacman_dx = self.pacman[0] - gx
                pacman_dy = self.pacman[1] - gy
                pacman_dist = abs(pacman_dx) + abs(pacman_dy)
                source = 2

            feat.extend([
                wall_dist / scale_hw,
                pacman_dist / scale_hw,
                pacman_dx / self.H,
                pacman_dy / self.W,
                source / 2.0,
            ])

        return np.array(feat, dtype=np.float32)

    def _ghost_can_see_pacman(self, ghost_pos):
        gx, gy = ghost_pos
        for dr, dc in self.RAYS:
            r, c = gx, gy
            while True:
                r += dr
                c += dc
                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break
                if (r, c) == self.pacman:
                    return True
        return False

    def get_obs(self):
        pacman_actor = self.pacman_obs()
        ghost_actor = np.stack(
            [self.ghost_obs_single(i) for i in range(len(self.ghosts))],
            axis=0
        )

        px, py = self.pacman

        pacman_rel = []
        for gx, gy in self.ghosts:
            pacman_rel.extend([
                (gx - px) / self.H,
                (gy - py) / self.W,
            ])
        pacman_rel = np.array(pacman_rel, dtype=np.float32)
        pacman_critic = np.concatenate([pacman_actor, pacman_rel], axis=0)

        ghost_critic = []
        for i, (gx, gy) in enumerate(self.ghosts):
            rel = np.array([
                (px - gx) / self.H,
                (py - gy) / self.W,
            ], dtype=np.float32)
            ghost_critic.append(np.concatenate([ghost_actor[i], rel], axis=0))
        ghost_critic = np.stack(ghost_critic, axis=0)

        return {
            "pacman_actor": pacman_actor,
            "pacman_critic": pacman_critic,
            "ghost_actor": ghost_actor,
            "ghost_critic": ghost_critic,
        }

    def step(self, pacman_action, ghost_actions):
        if self.done:
            raise RuntimeError("Episode ended. Call reset().")

        self.steps += 1

        for _ in range(self.pacman_speed):
            self.pacman = self.move(self.pacman, pacman_action)

        for _ in range(self.ghost_speed):
            self.ghosts = [self.move(g, a) for g, a in zip(self.ghosts, ghost_actions)]

        min_dist = min(self.manhattan(self.pacman, g) for g in self.ghosts)

        pacman_reward = -1.0 if min_dist <= 2 else 0.0
        ghost_rewards = np.array(
            [1.0 if self.manhattan(self.pacman, g) <= 2 else 0.0 for g in self.ghosts],
            dtype=np.float32
        )

        if self.steps >= self.max_steps:
            self.done = True

        obs = self.get_obs()
        rewards = {"pacman": pacman_reward, "ghosts": ghost_rewards}
        dones = {
            "pacman": float(self.done),
            "ghosts": np.array([float(self.done)] * len(self.ghosts), dtype=np.float32)
        }
        return obs, rewards, dones, {}

    def render_text(self):
        board = [row[:] for row in self.grid]
        pr, pc = self.pacman
        board[pr][pc] = "P"
        for i, (gr, gc) in enumerate(self.ghosts):
            board[gr][gc] = str(i + 1)
        print("\n".join("".join(row) for row in board))
        print(f"steps={self.steps}, done={self.done}")


# =========================================================
# 2. MODEL
# =========================================================

class RecurrentActorCritic(nn.Module):
    def __init__(self, actor_obs_dim, critic_obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.actor_encoder = nn.Sequential(
            nn.Linear(actor_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.critic_encoder = nn.Sequential(
            nn.Linear(critic_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.actor_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)
        self.critic_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)

        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def init_hidden(self, batch_size, device):
        actor_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        actor_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        return actor_h, actor_c, critic_h, critic_c

    def forward_step(self, actor_obs, critic_obs, hidden):
        actor_h, actor_c, critic_h, critic_c = hidden

        actor_x = self.actor_encoder(actor_obs).unsqueeze(0)
        critic_x = self.critic_encoder(critic_obs).unsqueeze(0)

        actor_out, (actor_h, actor_c) = self.actor_lstm(actor_x, (actor_h, actor_c))
        critic_out, (critic_h, critic_c) = self.critic_lstm(critic_x, (critic_h, critic_c))

        actor_out = actor_out.squeeze(0)
        critic_out = critic_out.squeeze(0)

        logits = self.policy_head(actor_out)
        value = self.value_head(critic_out).squeeze(-1)

        hidden = (actor_h, actor_c, critic_h, critic_c)
        return logits, value, hidden


# =========================================================
# 3. LOAD CHECKPOINT
# =========================================================

def load_checkpoint(path, pacman_net, ghost_net, device):
    ckpt = torch.load(path, map_location=device)

    pacman_net.load_state_dict(ckpt["pacman_model_state_dict"])
    ghost_net.load_state_dict(ckpt["ghost_model_state_dict"])

    print(f"[Loaded checkpoint] update={ckpt['update']}")
    return ckpt


# =========================================================
# 4. PLAY GAME
# =========================================================

@torch.no_grad()
def play_game(
    env,
    pacman_net,
    ghost_net,
    device="cpu",
    render=True,
    delay=0.5,
    stochastic=True,
    clear_screen=True,
    max_steps=None
):
    pacman_net.eval()
    ghost_net.eval()

    obs = env.reset()
    done = False

    pac_hidden = pacman_net.init_hidden(1, device)
    ghost_hidden = ghost_net.init_hidden(len(env.ghosts), device)

    step = 0
    pac_total = 0.0
    ghost_total = np.zeros(len(env.ghosts), dtype=np.float32)

    while not done:
        pac_actor_obs = torch.tensor(
            obs["pacman_actor"], dtype=torch.float32, device=device
        ).unsqueeze(0)
        pac_critic_obs = torch.tensor(
            obs["pacman_critic"], dtype=torch.float32, device=device
        ).unsqueeze(0)

        ghost_actor_obs = torch.tensor(
            obs["ghost_actor"], dtype=torch.float32, device=device
        )
        ghost_critic_obs = torch.tensor(
            obs["ghost_critic"], dtype=torch.float32, device=device
        )

        pac_logits, _, pac_hidden = pacman_net.forward_step(
            pac_actor_obs, pac_critic_obs, pac_hidden
        )
        ghost_logits, _, ghost_hidden = ghost_net.forward_step(
            ghost_actor_obs, ghost_critic_obs, ghost_hidden
        )

        if stochastic:
            pac_dist = Categorical(logits=pac_logits)
            pac_action = pac_dist.sample().item()

            ghost_dist = Categorical(logits=ghost_logits)
            ghost_actions = ghost_dist.sample().cpu().numpy().tolist()
        else:
            pac_action = torch.argmax(pac_logits, dim=-1).item()
            ghost_actions = torch.argmax(ghost_logits, dim=-1).cpu().numpy().tolist()

        obs, rewards, dones, _ = env.step(pac_action, ghost_actions)

        pac_total += rewards["pacman"]
        ghost_total += rewards["ghosts"]
        done = bool(dones["pacman"])

        if render:
            if clear_screen:
                clear_output(wait=True)

            print(f"[Step {step}]")
            env.render_text()
            print("Pacman reward:", rewards["pacman"])
            print("Ghost rewards:", rewards["ghosts"])
            print("Pacman total:", pac_total)
            print("Ghost mean total:", ghost_total.mean())
            print("-" * 40)
            time.sleep(delay)

        step += 1
        if max_steps is not None and step >= max_steps:
            break

    print("\n=== Final Result ===")
    print("Pacman total reward:", pac_total)
    print("Ghost mean total reward:", ghost_total.mean())


# =========================================================
# 5. RUN
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

env = TagMARLEnv(
    max_steps=200,
    seed=42,
    min_start_dist=4,
    pacman_speed=2,
    ghost_speed=1
)

pacman_net = RecurrentActorCritic(
    actor_obs_dim=40,
    critic_obs_dim=44,
    action_dim=5,
    hidden_dim=128
).to(device)

ghost_net = RecurrentActorCritic(
    actor_obs_dim=40,
    critic_obs_dim=42,
    action_dim=5,
    hidden_dim=128
).to(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu


In [46]:
# 체크포인트 경로 수정해서 사용
checkpoint_path = "/content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_900.pt"

load_checkpoint(checkpoint_path, pacman_net, ghost_net, device)

play_game(
    env,
    pacman_net,
    ghost_net,
    device=device,
    render=True,
    delay=0.2,
    stochastic=True,   # False로 바꾸면 greedy
    clear_screen=True,
    max_steps=200
)

[Step 37]
###########
#000000000#
#000#0P000#
#000#00#10#
#0#0#00#00#
#0#0#00#02#
#0#0#00#00#
#0#0#00#00#
#0#0#00000#
#000000000#
###########
steps=38, done=False
Pacman reward: 0.0
Ghost rewards: [0. 0.]
Pacman total: -12.0
Ghost mean total: 7.5
----------------------------------------


KeyboardInterrupt: 